In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
from pandas_datareader import data as web

series = {
    "DGS2": "2Y Treasury Yield",
    "DGS10": "10Y Treasury Yield",
    "VIXCLS": "VIX",
    "DFF": "Federal Funds Rate"
}

start_date = "2021-01-01"
end_date = "2025-12-31"

market_data = pd.DataFrame()

for ticker, name in series.items():
    df = web.DataReader(
        ticker,
        "fred",
        start_date,
        end_date
    )
    df = df.rename(columns={ticker: name})
    
    if market_data.empty:
        market_data = df
    else:
        market_data = market_data.join(df, how="outer")

market_data.head()

,2Y Treasury Yield,10Y Treasury Yield,VIX,Federal Funds Rate
DATE,,,,
2021-01-01,NaN,NaN,NaN,0.09
2021-01-02,NaN,NaN,NaN,0.09
2021-01-03,NaN,NaN,NaN,0.09
2021-01-04,0.11,0.93,26.97,0.09
2021-01-05,0.13,0.96,25.34,0.09


In [3]:
print("Rows:", len(market_data))
print("\nMissing values:")
print(market_data.isna().sum())

print("\nDate range:")
print(market_data.index.min(), "to", market_data.index.max())

Rows: 1826

Missing values:
2Y Treasury Yield     577
10Y Treasury Yield    577
VIX                   544
Federal Funds Rate      0
dtype: int64

Date range:
2021-01-01 00:00:00 to 2025-12-31 00:00:00


In [4]:
market_data = market_data.ffill()

print("Missing values after filling:")
print(market_data.isna().sum())

market_data.head()

Missing values after filling:
2Y Treasury Yield     3
10Y Treasury Yield    3
VIX                   3
Federal Funds Rate    0
dtype: int64


,2Y Treasury Yield,10Y Treasury Yield,VIX,Federal Funds Rate
DATE,,,,
2021-01-01,NaN,NaN,NaN,0.09
2021-01-02,NaN,NaN,NaN,0.09
2021-01-03,NaN,NaN,NaN,0.09
2021-01-04,0.11,0.93,26.97,0.09
2021-01-05,0.13,0.96,25.34,0.09


In [5]:
market_data = market_data.dropna(
    subset=["2Y Treasury Yield", "10Y Treasury Yield", "VIX"],
    how="all"
)

print("Rows after removing non trading days:", len(market_data))
print("\nRemaining missing values:")
print(market_data.isna().sum())

Rows after removing non trading days: 1823

Remaining missing values:
2Y Treasury Yield     0
10Y Treasury Yield    0
VIX                   0
Federal Funds Rate    0
dtype: int64


In [6]:
market_data.to_csv(
    "../data/market_data_clean.csv",
    index=True
)

print("Saved:", len(market_data), "market observations")

Saved: 1823 market observations


In [7]:
market_data.head()

,2Y Treasury Yield,10Y Treasury Yield,VIX,Federal Funds Rate
DATE,,,,
2021-01-04,0.11,0.93,26.97,0.09
2021-01-05,0.13,0.96,25.34,0.09
2021-01-06,0.14,1.04,25.07,0.09
2021-01-07,0.14,1.08,22.37,0.09
2021-01-08,0.14,1.13,21.56,0.09


In [8]:
market_data.index = pd.to_datetime(market_data.index)

fomc = pd.read_csv("../data/fomc_statements_clean.csv")
fomc["decision_date"] = pd.to_datetime(fomc["decision_date"])

fomc[["decision_date", "clean_text"]].head()

,decision_date,clean_text
0,2021-01-27,The Federal Reserve is committed to using its ...
1,2021-03-17,The Federal Reserve is committed to using its ...
2,2021-04-28,The Federal Reserve is committed to using its ...
3,2021-06-16,The Federal Reserve is committed to using its ...
4,2021-07-28,The Federal Reserve is committed to using its ...


In [9]:
fomc["decision_date"] = pd.to_datetime(fomc["decision_date"]).dt.normalize()

market_data.index = pd.to_datetime(market_data.index).normalize()

fomc_market = fomc.merge(
    market_data,
    left_on="decision_date",
    right_index=True,
    how="left"
)

fomc_market.head()

,decision_date,statement_date,clean_text,2Y Treasury Yield,10Y Treasury Yield,VIX,Federal Funds Rate
0,2021-01-27,2021-01-27,The Federal Reserve is committed to using its ...,0.12,1.04,37.21,0.08
1,2021-03-17,2021-03-17,The Federal Reserve is committed to using its ...,0.13,1.63,19.23,0.07
2,2021-04-28,2021-04-28,The Federal Reserve is committed to using its ...,0.17,1.63,17.28,0.07
3,2021-06-16,2021-06-16,The Federal Reserve is committed to using its ...,0.21,1.57,18.15,0.06
4,2021-07-28,2021-07-28,The Federal Reserve is committed to using its ...,0.20,1.26,18.31,0.10


In [10]:
print("Total FOMC meetings:", len(fomc_market))
print("\nMissing market data:")
print(fomc_market[["2Y Treasury Yield", "10Y Treasury Yield", "VIX", "Federal Funds Rate"]].isna().sum())

Total FOMC meetings: 40

Missing market data:
2Y Treasury Yield     0
10Y Treasury Yield    0
VIX                   0
Federal Funds Rate    0
dtype: int64


In [13]:
hawkish_words = [
    "inflation", "inflationary", "elevated", "persistent",
    "tighten", "tightening", "restrictive", "higher",
    "raise", "raising", "increase", "increases",
    "strong", "firm", "overheating"
]

dovish_words = [
    "accommodative", "accommodation", "support",
    "supportive", "lower", "lowering", "reduce",
    "reduced", "easing", "ease", "patient",
    "weak", "weaker", "uncertainty", "downside"
]

def tone_score(text):
    text = text.lower()
    
    hawkish = sum(text.count(word) for word in hawkish_words)
    dovish = sum(text.count(word) for word in dovish_words)
    
    return hawkish - dovish

fomc_market["tone_score"] = fomc_market["clean_text"].apply(tone_score)

fomc_market[["decision_date", "tone_score"]].head(10)

,decision_date,tone_score
0,2021-01-27,2
1,2021-03-17,4
2,2021-04-28,3
3,2021-06-16,0
4,2021-07-28,3
5,2021-09-22,4
6,2021-11-03,5
7,2021-12-15,-1
8,2022-01-26,1
9,2022-03-16,13


In [14]:
fomc_market["2Y_change"] = fomc_market["2Y Treasury Yield"].diff()
fomc_market["10Y_change"] = fomc_market["10Y Treasury Yield"].diff()
fomc_market["VIX_change"] = fomc_market["VIX"].diff()

fomc_market[
    ["decision_date", "tone_score", "2Y_change", "10Y_change", "VIX_change"]
].head(10)

,decision_date,tone_score,2Y_change,10Y_change,VIX_change
0,2021-01-27,2,NaN,NaN,NaN
1,2021-03-17,4,0.01,0.59,-17.98
2,2021-04-28,3,0.04,0.00,-1.95
3,2021-06-16,0,0.04,-0.06,0.87
4,2021-07-28,3,-0.01,-0.31,0.16
5,2021-09-22,4,0.05,0.06,2.56
6,2021-11-03,5,0.22,0.28,-5.77
7,2021-12-15,-1,0.22,-0.13,4.19
8,2022-01-26,1,0.44,0.38,12.67
9,2022-03-16,13,0.82,0.34,-5.29


In [15]:
fomc_market["2Y_next_day"] = fomc_market["2Y Treasury Yield"].shift(-1)
fomc_market["10Y_next_day"] = fomc_market["10Y Treasury Yield"].shift(-1)
fomc_market["VIX_next_day"] = fomc_market["VIX"].shift(-1)

fomc_market["2Y_reaction"] = (
    fomc_market["2Y_next_day"] - fomc_market["2Y Treasury Yield"]
)

fomc_market["10Y_reaction"] = (
    fomc_market["10Y_next_day"] - fomc_market["10Y Treasury Yield"]
)

fomc_market["VIX_reaction"] = (
    fomc_market["VIX_next_day"] - fomc_market["VIX"]
)

fomc_market[
    [
        "decision_date",
        "tone_score",
        "2Y_reaction",
        "10Y_reaction",
        "VIX_reaction"
    ]
].head(10)

,decision_date,tone_score,2Y_reaction,10Y_reaction,VIX_reaction
0,2021-01-27,2,0.01,0.59,-17.98
1,2021-03-17,4,0.04,0.00,-1.95
2,2021-04-28,3,0.04,-0.06,0.87
3,2021-06-16,0,-0.01,-0.31,0.16
4,2021-07-28,3,0.05,0.06,2.56
5,2021-09-22,4,0.22,0.28,-5.77
6,2021-11-03,5,0.22,-0.13,4.19
7,2021-12-15,-1,0.44,0.38,12.67
8,2022-01-26,1,0.82,0.34,-5.29
9,2022-03-16,13,0.71,0.74,-1.25


In [16]:
# Find the first trading day after each FOMC decision
next_day_data = []

for date in fomc_market["decision_date"]:
    future_dates = market_data.index[market_data.index > date]

    if len(future_dates) > 0:
        next_date = future_dates[0]

        row = market_data.loc[next_date]

        next_day_data.append({
            "decision_date": date,
            "next_trading_date": next_date,
            "2Y_next": row["2Y Treasury Yield"],
            "10Y_next": row["10Y Treasury Yield"],
            "VIX_next": row["VIX"]
        })

next_day_data = pd.DataFrame(next_day_data)

fomc_market = fomc_market.merge(
    next_day_data,
    on="decision_date",
    how="left"
)

fomc_market["2Y_reaction"] = (
    fomc_market["2Y_next"] -
    fomc_market["2Y Treasury Yield"]
)

fomc_market["10Y_reaction"] = (
    fomc_market["10Y_next"] -
    fomc_market["10Y Treasury Yield"]
)

fomc_market["VIX_reaction"] = (
    fomc_market["VIX_next"] -
    fomc_market["VIX"]
)

fomc_market[
    [
        "decision_date",
        "next_trading_date",
        "tone_score",
        "2Y_reaction",
        "10Y_reaction",
        "VIX_reaction"
    ]
].head(10)

,decision_date,next_trading_date,tone_score,2Y_reaction,10Y_reaction,VIX_reaction
0,2021-01-27,2021-01-28,2,0.00,0.03,-7.00
1,2021-03-17,2021-03-18,4,0.03,0.08,2.35
2,2021-04-28,2021-04-29,3,-0.01,0.02,0.33
3,2021-06-16,2021-06-17,0,0.02,-0.05,-0.40
4,2021-07-28,2021-07-29,3,0.00,0.02,-0.61
5,2021-09-22,2021-09-23,4,0.02,0.09,-2.24
6,2021-11-03,2021-11-04,5,-0.06,-0.07,0.34
7,2021-12-15,2021-12-16,-1,-0.05,-0.03,1.28
8,2022-01-26,2022-01-27,1,0.05,-0.04,-1.47
9,2022-03-16,2022-03-17,13,-0.01,0.01,-1.00


In [17]:
correlations = fomc_market[
    ["tone_score", "2Y_reaction", "10Y_reaction", "VIX_reaction"]
].corr()

correlations

,tone_score,2Y_reaction,10Y_reaction,VIX_reaction
tone_score,1.000000,-0.157578,-0.055171,0.331358
2Y_reaction,-0.157578,1.000000,0.701513,-0.094155
10Y_reaction,-0.055171,0.701513,1.000000,0.019379
VIX_reaction,0.331358,-0.094155,0.019379,1.000000


In [18]:
import statsmodels.api as sm

X = fomc_market[["tone_score"]]

X = sm.add_constant(X)

for y in ["2Y_reaction", "10Y_reaction", "VIX_reaction"]:
    model = sm.OLS(fomc_market[y], X).fit()

    print("\n" + "=" * 60)
    print("DEPENDENT VARIABLE:", y)
    print("=" * 60)
    print(model.summary())


DEPENDENT VARIABLE: 2Y_reaction
                            OLS Regression Results                            
Dep. Variable:            2Y_reaction   R-squared:                       0.025
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.9676
Date:                Sun, 09 Aug 2026   Prob (F-statistic):              0.332
Time:                        13:55:54   Log-Likelihood:                 50.885
No. Observations:                  40   AIC:                            -97.77
Df Residuals:                      38   BIC:                            -94.39
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.00